In [465]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
import matplotlib.pyplot as plt
import seaborn as sns


In [466]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(test.shape)
train.shape

(1459, 80)


(1460, 81)

In [445]:
total = train.isnull().sum().sort_values(ascending=False)
percent = (train.isnull().sum()/train.isnull().count().sort_values(ascending=False))
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)


,Total,Percent
PoolQC,1453,0.995205
MiscFeature,1406,0.963014
Alley,1369,0.937671
Fence,1179,0.807534
MasVnrType,872,0.597260
FireplaceQu,690,0.472603
LotFrontage,259,0.177397
GarageYrBlt,81,0.055479
GarageCond,81,0.055479
GarageType,81,0.055479


In [446]:
train = train.drop(columns=(missing_data[missing_data['Percent'] > 0.15]).index)
train_drop

Index(['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu',
       'LotFrontage'],
      dtype='object')

In [447]:
for col in ('GarageType', 'GarageFinish', 'GarageQual', 'GarageCond'):
    train[col] = train[col].fillna('None')
for col in ('GarageYrBlt', 'GarageArea', 'GarageCars'):
    train[col] = train[col].fillna(0)
for col in ('BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'BsmtFullBath', 'BsmtHalfBath'):
    train[col] = train[col].fillna(0)
for col in ('BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'):
    train[col] = train[col].fillna('None')


In [448]:
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])


In [449]:
train['GrLivArea'] = np.log(train['GrLivArea'])
train.loc[train['TotalBsmtSF'] > 0, 'TotalBsmtSF'] = np.log(train['TotalBsmtSF'])
train['MSSubClass'] = train['MSSubClass'].apply(str)
train['OverallCond'] = train['OverallCond'].astype(str)
train['YrSold'] = train['YrSold'].astype(str)
train['MoSold'] = train['MoSold'].astype(str)


c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\Users\arnov\AppData\Local\Temp\ipykernel_11664\904153424.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[6.75227038 7.14045304 6.82437367 ... 7.04925484 6.98286275 7.13568735]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train.loc[train['TotalBsmtSF'] > 0, 'TotalBsmtSF'] = np.log(train['TotalBsmtSF'])


In [450]:
cols = ('ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual', 'Functional', 'LandSlope',
        'LotShape', 'PavedDrive', 'CentralAir', 'MSSubClass', 'OverallCond', 'YrSold', 'MoSold')

for c in cols:
    lbl = LabelEncoder()
    lbl.fit(list(train[c].values))
    train[c] = lbl.transform(list(train[c].values))


In [451]:
train.shape

(1460, 74)

In [452]:
# Appliquer get_dummies séparément
train = pd.get_dummies(train)
test = pd.get_dummies(test)

missing_cols = set(train.columns) - set(test.columns)
for col in missing_cols:
    test[col] = 0  


test = test[train.columns]




In [453]:
train.shape

(1460, 248)

In [454]:
test.shape

(1459, 248)

In [455]:
y_train = train.SalePrice.values
train.drop(['SalePrice'], axis = 1, inplace = True)

In [456]:
n_folds = 5
def rmse_cv(model):
    kf = KFold(n_folds, shuffle=True, random_state=42).get_n_splits(train.values)
    rmse = np.sqrt(-cross_val_score(model, train.values, y_train, scoring='neg_mean_squared_error', cv=kf))
    return(rmse)

In [457]:
ENet = make_pipeline(RobustScaler(), ElasticNet(alpha=0.00005, l1_ratio=.9,max_iter=3000, random_state=3))
lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.00004,max_iter=3000, random_state=1))
KRR = KernelRidge(alpha=1, kernel='polynomial', gamma=0.4, degree=1, coef0=2.5)
GBoost = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                  max_depth=4, max_features='sqrt',
                                  min_samples_leaf=15, min_samples_split=10,
                                  loss='huber', random_state=5)


In [458]:
class AveragingModels(BaseEstimator, RegressorMixin, TransformerMixin):
    def __init__(self, models):
        self.models = models
    
    def fit(self, X, y):
        self.models_ = [clone(x) for x in self.models]  # Correct clone usage
        for model in self.models_:
            model.fit(X, y)
        return self

    def predict(self, X):
        predictions = np.column_stack([
            model.predict(X) for model in self.models_
        ])
        return np.mean(predictions, axis=1)


In [459]:



average_models = AveragingModels(models=(ENet, lasso, KRR, GBoost))

# Validation croisée pour le modèle d'ensemble
n_folds = 5

def rmse_cv(model):
    kf = KFold(n_folds, shuffle=True, random_state=42).get_n_splits(train.values)
    rmse = np.sqrt(-cross_val_score(model, train.values, y_train, 
                                    scoring='neg_mean_squared_error', cv=kf))
    return rmse

score = rmse_cv(average_models).mean()
print('Averaged score: {}'.format(round(score, 5)))


c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.705e+11, tolerance: 7.592e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.685e+11, tolerance: 7.592e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

Averaged score: 30429.2445


In [460]:
# Entraîner le modèle sur tout le dataset d'entraînement
average_models.fit(train.values, y_train)


c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.320e+11, tolerance: 9.208e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.296e+11, tolerance: 9.208e+08
  model = cd_fast.enet_coordinate_descent(


AveragingModels(models=(Pipeline(steps=[('robustscaler', RobustScaler()),
                                        ('elasticnet',
                                         ElasticNet(alpha=5e-05, l1_ratio=0.9,
                                                    max_iter=3000,
                                                    random_state=3))]),
                        Pipeline(steps=[('robustscaler', RobustScaler()),
                                        ('lasso',
                                         Lasso(alpha=4e-05, max_iter=3000,
                                               random_state=1))]),
                        KernelRidge(coef0=2.5, degree=1, gamma=0.4,
                                    kernel='polynomial'),
                        GradientBoostingRegressor(learning_rate=0.05,
                                                  loss='huber', max_depth=4,
                                                  max_features='sqrt',
                                                  min_samples_leaf=15,
                                                  min_samples_split=10,
                                                  n_estimators=300,
                                                  random_state=5)))

In [463]:
# S'assurer que la colonne 'SalePrice' n'est pas dans test (elle n'est pas censée être présente)
if 'SalePrice' in test.columns:
    test = test.drop('SalePrice', axis=1)

# Réordonner les colonnes de test pour correspondre à celles de train
test = test[train.columns]

# Ensuite, tu peux faire les prédictions
predictions = average_models.predict(test)


ValueError: Input X contains NaN.
ElasticNet does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [392]:
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999